In [1]:
!pip install -U google-generativeai

In [10]:
import os
import sqlite3
import json
import time
from PIL import Image
from tqdm.auto import tqdm
from google.colab import userdata

# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 최신버전 Gemini SDK 임포트
from google import genai
from google.genai import types

# API 키 세팅 및 최신 클라이언트 생성
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

DB_PATH = "/content/drive/MyDrive/CV_Project/seongdong_db/metadata_seongdong.db"
OUTPUT_FILE = "/content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_gemini_finetune.jsonl"
BATCH_SIZE = 10

def process_image_batch(batch_rows):
    contents = []

    instruction = f"""
    당신은 프롭테크 입지 분석 AI입니다. 다음 제공되는 {len(batch_rows)}장의 위성 사진들을 각각 1문장으로 묘사하세요.
    수치를 쓰지 말고 건물의 밀집도, 녹지, 도로 형태 등 시각적 분위기를 묘사하세요.
    결과는 반드시 아래의 JSON Array 포맷으로만 출력하세요:
    [
      {{"file_name": "이미지파일명1", "text": "묘사 내용"}},
      {{"file_name": "이미지파일명2", "text": "묘사 내용"}}
    ]
    """
    contents.append(instruction)

    valid_rows = []

    BASE_DIR = "/content/drive/MyDrive/CV_Project"

    for row in batch_rows:
        tile_id, district, db_image_path = row

        clean_path = db_image_path.replace("\\", "/")
        image_path = os.path.join(BASE_DIR, clean_path)

        if not os.path.exists(image_path):
            print(f"⚠️ 파일 없음 스킵: {image_path}") # 혹시 파일이 없으면 이유를 출력
            continue

        try:
            img = Image.open(image_path).convert('RGB')
            contents.append(f"파일명: {os.path.basename(image_path)}, 위치: {district}")
            contents.append(img)
            valid_rows.append(row)
        except Exception as e:
            print(f"이미지 로드 실패 ({image_path}): {e}")

    if not valid_rows:
        return []

    MAX_RETRIES = 5
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.2,
                )
            )
            results = json.loads(response.text)
            return results

        except Exception as e:
            error_str = str(e)
            if '503' in error_str or 'UNAVAILABLE' in error_str:
                wait = (attempt + 1) * 10  # 10초, 20초, 30초...
                print(f"⏳ 503 에러, {wait}초 후 재시도... ({attempt+1}/{MAX_RETRIES})")
                time.sleep(wait)
            else:
                print(f"\n🚨 API 호출 또는 파싱 에러: {e}")
                return []

    print("❌ 최대 재시도 횟수 초과, 배치 스킵")
    return []

def main():
    if not GEMINI_API_KEY:
        print("❌ GEMINI_API_KEY가 설정되지 않았습니다.")
        return

    print("1. SQLite DB에서 이미지 목록 로드 중...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT tile_id, district, image_path FROM tile_info")

    rows = cursor.fetchall()
    conn.close()

    print(f"2. 총 {len(rows)}장 타일: 한 번에 {BATCH_SIZE}장씩 최신 Gemini API로 다중 분석 시작...")

    all_results = []

    for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="배치 처리 중"):
        batch_rows = rows[i:i+BATCH_SIZE]

        batch_results = process_image_batch(batch_rows)
        if batch_results:
            all_results.extend(batch_results)

        time.sleep(5)

    print(f"\n3. 추출 완료! 총 {len(all_results)}개의 캡션 결과를 저장합니다...")
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for res in all_results:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

    print(f"🎉 성공적으로 최신 API를 통해 데이터셋이 생성되었습니다: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. SQLite DB에서 이미지 목록 로드 중...
2. 총 1824장 타일: 한 번에 10장씩 최신 Gemini API로 다중 분석 시작...


배치 처리 중:   0%|          | 0/183 [00:00<?, ?it/s]

⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)

🚨 API 호출 또는 파싱 에러: Expecting ':' delimiter: line 9 column 31 (char 792)

🚨 API 호출 또는 파싱 에러: Expecting ':' delimiter: line 7 column 31 (char 613)

3. 추출 완료! 총 1804개의 캡션 결과를 저장합니다...
🎉 성공적으로 최신 API를 통해 데이터셋이 생성되었습니다: /content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_gemini_finetune.jsonl
